
# Telco Customer Churn: Retention Risk Analysis

**Business Problem:** The company wants to know which customers are most likely
to cancel their subscription, and what's actually driving that risk price,
contract type, or service experience so they can prioritize a retention
intervention that's worth the cost.

**Hypothesis:** Customers on month-to-month contracts with higher monthly
charges and no add-on services (like tech support or online security) are
far more likely to churn because they have less "lock-in" and less
perceived value for the price.

In [1]:
import pandas as pd
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')
df.head()

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


## Data Cleaning

In [2]:
# Check for missing values and duplicates
df.isnull().sum()

,0
customerID,0
gender,0
SeniorCitizen,0
Partner,0
Dependents,0
tenure,0
PhoneService,0
MultipleLines,0
InternetService,0
OnlineSecurity,0


In [3]:
# Check data types
df.dtypes

,0
customerID,object
gender,object
SeniorCitizen,int64
Partner,object
Dependents,object
tenure,int64
PhoneService,object
MultipleLines,object
InternetService,object
OnlineSecurity,object


In [4]:
# TotalCharges is stored as text (object) instead of numbers.
# Check how many values fail to convert:
pd.to_numeric(df['TotalCharges'], errors='coerce').isnull().sum()

np.int64(11)

In [5]:
# Investigate why: check tenure for these problem rows
df[pd.to_numeric(df['TotalCharges'], errors='coerce').isnull()][['tenure','MonthlyCharges','TotalCharges']]

,tenure,MonthlyCharges,TotalCharges
488,0,52.55,
753,0,20.25,
936,0,80.85,
1082,0,25.75,
1340,0,56.05,
3331,0,19.85,
3826,0,25.35,
4380,0,20.00,
5218,0,19.70,
6670,0,73.35,


In [6]:
# All 11 are brand-new customers (tenure = 0), so no bill has
# completed yet. Fill with 0 to reflect reality, not missing data.
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Confirm the fix
print(df['TotalCharges'].isnull().sum())
print(df.dtypes['TotalCharges'])

0
float64


In [7]:
# Check for duplicate rows
df.duplicated().sum()

np.int64(0)


## Exploratory Analysis

In [8]:
# Overall churn rate
df['Churn'].value_counts(normalize=True)

,proportion
Churn,
No,0.73463
Yes,0.26537


In [9]:
# Churn rate by contract type
df.groupby('Contract')['Churn'].value_counts(normalize=True)

Contract        Churn
Month-to-month  No       0.572903
                Yes      0.427097
One year        No       0.887305
                Yes      0.112695
Two year        No       0.971681
                Yes      0.028319
Name: proportion, dtype: float64

In [10]:
# Average monthly charge: churned vs retained
df.groupby('Churn')['MonthlyCharges'].mean()

,MonthlyCharges
Churn,
No,61.265124
Yes,74.441332


In [11]:
# Average monthly charge, controlling for contract type
df.groupby(['Contract', 'Churn'])['MonthlyCharges'].mean()

Contract        Churn
Month-to-month  No       61.462635
                Yes      73.019396
One year        No       62.508148
                Yes      85.050904
Two year        No       60.012477
                Yes      86.777083
Name: MonthlyCharges, dtype: float64

In [12]:
# Add-on service adoption: churned vs retained
df.groupby('Churn')[['OnlineSecurity', 'TechSupport', 'StreamingTV', 'StreamingMovies']].apply(lambda x: (x == 'Yes').mean())

,OnlineSecurity,TechSupport,StreamingTV,StreamingMovies
Churn,,,,
No,0.333204,0.335137,0.365868,0.369927
Yes,0.157838,0.165864,0.435527,0.437667



## Key Findings

1. **Overall churn rate is 27%** — high for a subscription business, confirming
a real retention problem worth investigating.

2. **Contract type is the strongest lever**: month-to-month customers churn at
43%, compared to just 3% for two-year contracts  roughly a 15x difference.
Contract length functions as a lock-in mechanism.

3. **Price matters independently of contract type**: even within the same
contract length, churned customers paid more than retained customers and
this gap widens on longer contracts (up to +$27/month on two-year plans),
suggesting price dissatisfaction can outweigh contractual lock-in.

4. **Not all add-ons reduce churn equally**: online security and tech support
are strongly linked to retention (customers who stayed had these ~2x more
often), while streaming TV/movies show the opposite pattern churned
customers had these slightly more often. This suggests reliability/support
services create genuine dependency, while entertainment add-ons don't.


## Recommendation

Rather than pushing month-to-month customers into longer contracts which
reduces churn numbers without addressing why customers want to leave the
company should proactively target its highest-risk segment (month-to-month,
above-average monthly charges, no tech support or security add-ons) with a
free trial of tech support and security services. These add-ons showed a
strong link to retention in this data, likely because they create genuine
day-to-day reliance rather than contractual pressure. This approach builds
loyalty through real value instead of exit friction more sustainable than
locking in unhappy customers who leave the moment they're able to.


## Limitations

This is correlational, not causal we can't say for certain that adding
tech support *causes* people to stay, only that the two are linked. It's
possible customers who already intend to stay longer are simply more willing
to try add-ons in the first place (reverse causality). To properly validate
this recommendation, the company would ideally run a controlled test: offer
free tech support trials to a random sample of at-risk customers and compare
their churn rate to a similar group that didn't get the offer, over a few
months.